# gold_hospital_performance_model

**Project:** Australian Hospital Performance & Operations Analytics Platform  
**Layer:** Gold  
**Input:** Silver Lakehouse tables  
**Output:** `gold_fact_measure_value`, `gold_dim_reporting_unit`, `gold_dim_measure`, `gold_dim_period`, `gold_dim_reported_measure`

This notebook builds the Gold reporting model from the existing Silver tables using a Kimball-style star schema.

**Grain:** One row in `gold_fact_measure_value` represents one source observation for a reporting unit, measure, reporting period and reported-measure variant.

The fact table links to four dimensions: reporting unit, measure, period and reported measure. Source identifiers and suppression/caveat details are kept for traceability. Missing values remain missing; they are not changed to zero.

The transformations and output table structures are kept as they are so the existing semantic model and report continue to work.


In [1]:
# Set the input and output table names and define small helper functions.

from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

SILVER_OBSERVATIONS = "silver_hospital_measure_observations"
SILVER_MEASURES = "silver_measures_ref"
SILVER_CATEGORIES = "silver_measure_categories"
SILVER_DATASETS = "silver_datasets"
SILVER_REPORTING_UNITS = "silver_reporting_units"

GOLD_FACT = "gold_fact_measure_value"
GOLD_REPORTING_UNITS = "gold_dim_reporting_unit"
GOLD_MEASURES = "gold_dim_measure"
GOLD_PERIODS = "gold_dim_period"
GOLD_REPORTED_MEASURES = "gold_dim_reported_measure"

def require_columns(df, required, label):
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f"{label} is missing required columns: {missing}")

def first_existing(columns, candidates, label):
    for name in candidates:
        if name in columns:
            return name
    raise ValueError(f"Could not find {label}. Tried {candidates}; available columns: {columns}")

def assert_unique(df, columns, label):
    duplicates = df.groupBy(*columns).count().filter(F.col("count") > 1)
    if duplicates.limit(1).count():
        display(duplicates.limit(20))
        raise ValueError(f"{label} is not unique on {columns}.")

def add_surrogate_key(df, key_name, order_columns):
    # Stable ordering makes keys reproducible for a fixed set of business keys.
    w = Window.orderBy(*[F.col(c).asc_nulls_first() for c in order_columns])
    return df.withColumn(key_name, F.row_number().over(w).cast("long"))

def optional_col(df, name, dtype):
    if name in df.columns:
        return F.col(name).cast(dtype)
    return F.lit(None).cast(dtype)

print("Gold configuration loaded.")


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 3, Finished, Available, Finished, False)

Gold configuration loaded.


In [2]:
# Read the Silver tables and check that the columns needed for Gold are present.

silver_observations = spark.table(SILVER_OBSERVATIONS)
silver_measures = spark.table(SILVER_MEASURES)
silver_categories = spark.table(SILVER_CATEGORIES)
silver_datasets = spark.table(SILVER_DATASETS)
silver_reporting_units = spark.table(SILVER_REPORTING_UNITS)

require_columns(silver_observations, [
    "data_set_id", "measure_code", "reported_measure_code", "value",
    "reporting_unit_code", "reporting_start_date", "reporting_end_date"
], "Silver observations")
require_columns(silver_measures, ["measure_code", "measure_name"], "Silver measures")
require_columns(silver_datasets, ["data_set_id", "reporting_start_date", "reporting_end_date"], "Silver datasets")
require_columns(silver_reporting_units, ["reporting_unit_code", "reporting_unit_name"], "Silver reporting units")

print(f"Silver observation rows: {silver_observations.count():,}")
print(f"Silver measures: {silver_measures.count():,}")
print(f"Silver datasets: {silver_datasets.count():,}")
print(f"Silver reporting units: {silver_reporting_units.count():,}")
print("=== Observation schema ===")
silver_observations.printSchema()
print("=== Reporting-unit schema ===")
silver_reporting_units.printSchema()
print("=== Measure schema ===")
silver_measures.printSchema()
print("=== Dataset schema ===")
silver_datasets.printSchema()


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 4, Finished, Available, Finished, False)

Silver observation rows: 1,709,311
Silver measures: 33
Silver datasets: 7,244
Silver reporting units: 1,428
=== Observation schema ===
root
 |-- data_set_id: long (nullable = true)
 |-- source_measure_code: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- caveats: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- caveat_code: string (nullable = true)
 |    |    |-- caveat_display_value: string (nullable = true)
 |    |    |-- caveat_footnote: string (nullable = true)
 |    |    |-- caveat_name: string (nullable = true)
 |-- group_number: integer (nullable = true)
 |-- lower_value: double (nullable = true)
 |-- measure_code: string (nullable = true)
 |-- peer_group_summary: struct (nullable = true)
 |    |-- reporting_unit_code: string (nullable = true)
 |    |-- reporting_unit_name: string (nullable = true)
 |    |-- reporting_unit_type: struct (nullable = true)
 |    |    |-- reporting_unit_type_code: string (nullable = tru

In [3]:
# Build the reporting-unit dimension, including state, LHN and PHN mappings.

ru = silver_reporting_units
if "reporting_unit_type" in ru.columns and isinstance(ru.schema["reporting_unit_type"].dataType, T.StructType):
    type_fields = ru.schema["reporting_unit_type"].dataType.fieldNames()
    type_code_field = first_existing(type_fields, ["reporting_unit_type_code", "type_code"], "reporting-unit type code")
    type_name_field = first_existing(type_fields, ["reporting_unit_type_name", "type_name"], "reporting-unit type name")
    type_code = F.col(f"reporting_unit_type.{type_code_field}").cast("string")
    type_name = F.col(f"reporting_unit_type.{type_name_field}").cast("string")
else:
    type_code_col = first_existing(ru.columns, ["reporting_unit_type_code", "type_code"], "reporting-unit type code")
    type_name_col = first_existing(ru.columns, ["reporting_unit_type_name", "type_name"], "reporting-unit type name")
    type_code, type_name = F.col(type_code_col).cast("string"), F.col(type_name_col).cast("string")

def mapped_attribute(df, mapping_col, mapping_code, attribute):
    # Extract only documented mappings that are present in this snapshot.
    if mapping_col not in df.columns:
        return F.lit(None).cast("string")
    dt = df.schema[mapping_col].dataType
    if not isinstance(dt, T.ArrayType) or not isinstance(dt.elementType, T.StructType):
        return F.lit(None).cast("string")
    item_fields = dt.elementType.fieldNames()
    if not {"map_type", "mapped_reporting_unit"}.issubset(item_fields):
        return F.lit(None).cast("string")
    map_fields = dt.elementType["map_type"].dataType.fieldNames()
    value_fields = dt.elementType["mapped_reporting_unit"].dataType.fieldNames()
    if "mapped_reporting_unit_code" not in map_fields or attribute not in value_fields:
        return F.lit(None).cast("string")
    matches = F.filter(F.col(mapping_col), lambda x: x["map_type"]["mapped_reporting_unit_code"] == F.lit(mapping_code))
    return F.element_at(matches, 1)["mapped_reporting_unit"][attribute].cast("string")

mapping_col = "mapped_reporting_units" if "mapped_reporting_units" in ru.columns else None
ru_base = ru.select(
    F.col("reporting_unit_code").cast("string").alias("reporting_unit_code"),
    F.col("reporting_unit_name").cast("string").alias("reporting_unit_name"),
    type_code.alias("reporting_unit_type_code"), type_name.alias("reporting_unit_type_name"),
    optional_col(ru, "latitude", "double").alias("latitude"),
    optional_col(ru, "longitude", "double").alias("longitude"),
    optional_col(ru, "private", "boolean").alias("is_private"),
    optional_col(ru, "closed", "boolean").alias("is_closed"),
    mapped_attribute(ru, mapping_col, "STATE_MAPPING", "reporting_unit_code").alias("state_code"),
    mapped_attribute(ru, mapping_col, "STATE_MAPPING", "reporting_unit_name").alias("state_name"),
    mapped_attribute(ru, mapping_col, "H_LHN", "reporting_unit_code").alias("lhn_code"),
    mapped_attribute(ru, mapping_col, "H_LHN", "reporting_unit_name").alias("lhn_name"),
    mapped_attribute(ru, mapping_col, "H_PHN", "reporting_unit_code").alias("phn_code"),
    mapped_attribute(ru, mapping_col, "H_PHN", "reporting_unit_name").alias("phn_name")
).filter(F.col("reporting_unit_code").isNotNull())
assert_unique(ru_base, ["reporting_unit_code"], "Reporting-unit source")
ru_keyed = add_surrogate_key(ru_base, "reporting_unit_key", ["reporting_unit_code"])
unknown_ru = spark.createDataFrame(
    [(-1, "UNKNOWN", "Unknown reporting unit", None, None, None, None, None, None, None, None, None, None, None, None)],
    "reporting_unit_key long, reporting_unit_code string, reporting_unit_name string, "
    "reporting_unit_type_code string, reporting_unit_type_name string, latitude double, longitude double, "
    "is_private boolean, is_closed boolean, state_code string, state_name string, lhn_code string, lhn_name string, "
    "phn_code string, phn_name string"
)
ru_keyed = ru_keyed.select(
    "reporting_unit_key", "reporting_unit_code", "reporting_unit_name", "reporting_unit_type_code",
    "reporting_unit_type_name", "latitude", "longitude", "is_private", "is_closed",
    "state_code", "state_name", "lhn_code", "lhn_name", "phn_code", "phn_name"
)
dim_reporting_unit = unknown_ru.unionByName(ru_keyed)
assert_unique(dim_reporting_unit, ["reporting_unit_key"], "Reporting-unit dimension")
assert_unique(dim_reporting_unit.filter(F.col("reporting_unit_key") != -1), ["reporting_unit_code"], "Reporting-unit business key")
print(f"{GOLD_REPORTING_UNITS}: {dim_reporting_unit.count():,} rows including Unknown")
display(dim_reporting_unit.groupBy("reporting_unit_type_code", "reporting_unit_type_name").count().orderBy("reporting_unit_type_code"))


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 5, Finished, Available, Finished, False)

gold_dim_reporting_unit: 1,429 rows including Unknown


SynapseWidget(Synapse.DataFrame, ee4d2c9a-33b7-4b55-8f16-7969205407e3)

In [4]:
# Build the measure dimension with category and unit details.

m = silver_measures
cat = silver_categories
if "measure_categories" in m.columns and isinstance(m.schema["measure_categories"].dataType, T.ArrayType):
    cat_fields = m.schema["measure_categories"].dataType.elementType.fieldNames()
    category_expr = F.element_at(F.col("measure_categories"), 1)["measure_category_code"] if "measure_category_code" in cat_fields else F.lit(None)
elif "measure_category_code" in m.columns:
    category_expr = F.col("measure_category_code")
else:
    category_expr = F.lit(None)
cat_code = first_existing(cat.columns, ["measure_category_code", "category_code"], "category code")
cat_name = first_existing(cat.columns, ["measure_category_name", "category_name"], "category name")
cat_lookup = cat.select(F.col(cat_code).cast("string").alias("_category_code"), F.col(cat_name).cast("string").alias("measure_category_name")).dropDuplicates(["_category_code"])

unit_type = m.schema["units"].dataType if "units" in m.columns else None
if isinstance(unit_type, T.StructType):
    uf = unit_type.fieldNames()
    unit_name = F.col("units.units_name") if "units_name" in uf else F.lit(None)
    unit_display = F.col("units.units_display") if "units_display" in uf else F.lit(None)
    decimal_places = F.col("units.decimal_places") if "decimal_places" in uf else F.lit(None)
    units_prefix = F.col("units.units_are_prefix") if "units_are_prefix" in uf else F.lit(None)
else:
    unit_name = unit_display = decimal_places = units_prefix = F.lit(None)

measure_base = m.select(
    F.col("measure_code").cast("string").alias("measure_code"),
    F.col("measure_name").cast("string").alias("measure_name"),
    category_expr.cast("string").alias("measure_category_code"),
    unit_name.cast("string").alias("units_name"), unit_display.cast("string").alias("units_display"),
    decimal_places.cast("integer").alias("decimal_places"), units_prefix.cast("boolean").alias("units_are_prefix")
).join(cat_lookup, F.col("measure_category_code") == F.col("_category_code"), "left").drop("_category_code")
measure_base = measure_base.filter(F.col("measure_code").isNotNull())
assert_unique(measure_base, ["measure_code"], "Measure dimension source")
dim_measure = add_surrogate_key(measure_base, "measure_key", ["measure_code"]).select(
    "measure_key", "measure_code", "measure_name", "measure_category_code", "measure_category_name",
    "units_name", "units_display", "decimal_places", "units_are_prefix"
)
unknown_measure = spark.createDataFrame(
    [(-1, "UNKNOWN", "Unknown measure", None, None, None, None, None, None)],
    "measure_key long, measure_code string, measure_name string, measure_category_code string, "
    "measure_category_name string, units_name string, units_display string, decimal_places int, units_are_prefix boolean"
)
dim_measure = unknown_measure.unionByName(dim_measure)
assert_unique(dim_measure, ["measure_key"], "Measure dimension")
assert_unique(dim_measure.filter(F.col("measure_key") != -1), ["measure_code"], "Measure business key")
print(f"{GOLD_MEASURES}: {dim_measure.count():,} rows including Unknown")
display(dim_measure.orderBy("measure_key"))


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 6, Finished, Available, Finished, False)

gold_dim_measure: 34 rows including Unknown


SynapseWidget(Synapse.DataFrame, f33b406a-4994-450d-9d5e-a6425fc6395e)

In [5]:
# Build the reporting-period dimension from the dataset start and end dates.

dataset_dates = silver_datasets.select(
    F.col("data_set_id").cast("long").alias("data_set_id"),
    F.to_date("reporting_start_date").alias("reporting_start_date"),
    F.to_date("reporting_end_date").alias("reporting_end_date")
)
if dataset_dates.filter(F.col("data_set_id").isNull()).limit(1).count():
    raise ValueError("Dataset catalog contains a null data_set_id.")
if dataset_dates.groupBy("data_set_id").count().filter(F.col("count") > 1).limit(1).count():
    raise ValueError("Dataset catalog contains duplicate data_set_id values.")

periods = dataset_dates.select(
    F.col("reporting_start_date").alias("period_start_date"),
    F.col("reporting_end_date").alias("period_end_date")
).distinct().filter(F.col("period_start_date").isNotNull() & F.col("period_end_date").isNotNull())
start, end = F.col("period_start_date"), F.col("period_end_date")
is_fy = (F.month(start) == 7) & (F.month(end) == 6)
periods = periods.withColumn(
    "financial_year", F.when(is_fy, F.concat(F.year(start).cast("string"), F.lit("-"), F.substring(F.year(end).cast("string"), 3, 2)))
).withColumn(
    "period_type", F.when(is_fy, F.lit("Financial year")).otherwise(F.lit("Reporting period"))
).withColumn(
    "period_label", F.when(is_fy, F.col("financial_year")).otherwise(
        F.concat(F.date_format(start, "dd MMM yyyy"), F.lit(" to "), F.date_format(end, "dd MMM yyyy")))
).withColumn("period_sort_year", F.year(start))
assert_unique(periods, ["period_start_date", "period_end_date"], "Period dimension source")
dim_period = add_surrogate_key(periods, "period_key", ["period_start_date", "period_end_date"]).select(
    "period_key", "period_label", "period_type", "financial_year", "period_sort_year", "period_start_date", "period_end_date"
)
unknown_period = spark.createDataFrame(
    [(-1, "Unknown period", "Unknown", None, None, None, None)],
    "period_key long, period_label string, period_type string, financial_year string, period_sort_year int, period_start_date date, period_end_date date"
)
dim_period = unknown_period.unionByName(dim_period)
assert_unique(dim_period, ["period_key"], "Period dimension")
assert_unique(dim_period.filter(F.col("period_key") != -1), ["period_start_date", "period_end_date"], "Period business key")
print(f"{GOLD_PERIODS}: {dim_period.count():,} rows including Unknown")
display(dim_period.orderBy("period_start_date", "period_end_date"))


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 7, Finished, Available, Finished, False)

gold_dim_period: 61 rows including Unknown


SynapseWidget(Synapse.DataFrame, 13076a71-d121-4dd7-bd9d-93aca1a3b876)

In [6]:
# Build the reported-measure dimension and flag clear total variants.

if "reported_measure_summary" in silver_datasets.columns and isinstance(silver_datasets.schema["reported_measure_summary"].dataType, T.StructType):
    rm_fields = silver_datasets.schema["reported_measure_summary"].dataType.fieldNames()
    if "reported_measure_code" in rm_fields:
        rm_code_expr = F.col("reported_measure_summary.reported_measure_code").cast("string")
        rm_name_expr = F.col("reported_measure_summary.reported_measure_name").cast("string") if "reported_measure_name" in rm_fields else F.lit(None).cast("string")
        rm_names = silver_datasets.select(rm_code_expr.alias("reported_measure_code"), rm_name_expr.alias("reported_measure_name"))             .filter(F.col("reported_measure_code").isNotNull()).dropDuplicates()
        name_conflicts = rm_names.groupBy("reported_measure_code").agg(F.countDistinct("reported_measure_name").alias("n" )).filter(F.col("n") > 1)
        if name_conflicts.limit(1).count():
            display(name_conflicts.limit(20))
            raise ValueError("A reported-measure code maps to multiple names in dataset metadata.")
        rm_names = rm_names.dropDuplicates(["reported_measure_code"])
    else:
        rm_names = spark.createDataFrame([], "reported_measure_code string, reported_measure_name string")
else:
    rm_names = spark.createDataFrame([], "reported_measure_code string, reported_measure_name string")

rm_codes = silver_observations.select(F.col("reported_measure_code").cast("string").alias("reported_measure_code"))     .filter(F.col("reported_measure_code").isNotNull()).distinct()
rm_base = rm_codes.join(rm_names, "reported_measure_code", "left")
label = F.lower(F.trim(F.col("reported_measure_name")))
rm_base = rm_base.withColumn("is_total", F.coalesce((label == "total") | (label == "all") | label.startswith("all "), F.lit(False)))
dim_reported = add_surrogate_key(rm_base, "reported_measure_key", ["reported_measure_code"]).select(
    "reported_measure_key", "reported_measure_code", "reported_measure_name", "is_total"
)
unknown_rm = spark.createDataFrame(
    [(-1, "UNKNOWN", "Unknown reported-measure variant", False)],
    "reported_measure_key long, reported_measure_code string, reported_measure_name string, is_total boolean"
)
dim_reported_measure = unknown_rm.unionByName(dim_reported)
assert_unique(dim_reported_measure, ["reported_measure_key"], "Reported-measure dimension")
assert_unique(dim_reported_measure.filter(F.col("reported_measure_key") != -1), ["reported_measure_code"], "Reported-measure business key")
print(f"{GOLD_REPORTED_MEASURES}: {dim_reported_measure.count():,} rows including Unknown")
display(dim_reported_measure.orderBy("reported_measure_key"))


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 8, Finished, Available, Finished, False)

gold_dim_reported_measure: 691 rows including Unknown


SynapseWidget(Synapse.DataFrame, 43a7eba5-6598-4cc8-b0cc-99c1a5da4733)

In [7]:
# Join the Silver observations to the dimensions and prepare the Gold fact table.

source = silver_observations.withColumn("_period_start_date", F.to_date("reporting_start_date"))     .withColumn("_period_end_date", F.to_date("reporting_end_date"))

unit_lookup = dim_reporting_unit.filter(F.col("reporting_unit_key") != -1).select("reporting_unit_key", "reporting_unit_code")
measure_lookup = dim_measure.filter(F.col("measure_key") != -1).select("measure_key", "measure_code")
reported_lookup = dim_reported_measure.filter(F.col("reported_measure_key") != -1).select("reported_measure_key", "reported_measure_code")
period_lookup = dim_period.filter(F.col("period_key") != -1).select("period_key", "period_start_date", "period_end_date")

j = source.alias("s").join(unit_lookup.alias("u"), F.col("s.reporting_unit_code").cast("string") == F.col("u.reporting_unit_code"), "left")     .join(measure_lookup.alias("m"), F.col("s.measure_code").cast("string") == F.col("m.measure_code"), "left")     .join(reported_lookup.alias("r"), F.col("s.reported_measure_code").cast("string") == F.col("r.reported_measure_code"), "left")     .join(period_lookup.alias("p"), (F.col("s._period_start_date") == F.col("p.period_start_date")) & (F.col("s._period_end_date") == F.col("p.period_end_date")), "left")

source_count, joined_count = source.count(), j.count()
print(f"Silver fact-source rows: {source_count:,}")
print(f"Rows after dimension lookup joins: {joined_count:,}")
if joined_count != source_count:
    raise ValueError("Dimension joins changed the fact row count; check dimension key uniqueness.")

unmatched = j.select(
    F.sum((F.col("s.reporting_unit_code").isNotNull() & F.col("u.reporting_unit_key").isNull()).cast("long")).alias("reporting_units"),
    F.sum((F.col("s.measure_code").isNotNull() & F.col("m.measure_key").isNull()).cast("long")).alias("measures"),
    F.sum((F.col("s.reported_measure_code").isNotNull() & F.col("r.reported_measure_key").isNull()).cast("long")).alias("reported_measures"),
    F.sum((F.col("s._period_start_date").isNotNull() & F.col("s._period_end_date").isNotNull() & F.col("p.period_key").isNull()).cast("long")).alias("periods")
).first().asDict()
print("Non-null business keys without dimension matches:", unmatched)
if any((v or 0) > 0 for v in unmatched.values()):
    raise ValueError("At least one non-null fact business key did not resolve to a Gold dimension.")

suppression_count = F.col("s.suppression_count").cast("integer") if "suppression_count" in source.columns else F.lit(0).cast("integer")
caveat_count = F.col("s.caveat_count").cast("integer") if "caveat_count" in source.columns else F.lit(0).cast("integer")
is_suppressed = F.col("s.is_suppressed").cast("boolean") if "is_suppressed" in source.columns else suppression_count > 0
is_missing = F.col("s.is_value_missing").cast("boolean") if "is_value_missing" in source.columns else F.col("s.value").isNull()

# Keep the original structured reasons from Silver. Their code, name, display
# text and footnote fields help explain why a value is unavailable.
suppression_details = F.col("s.suppressions") if "suppressions" in source.columns else F.lit(None)
caveat_details = F.col("s.caveats") if "caveats" in source.columns else F.lit(None)

suppression_fields = []
if "suppressions" in source.columns:
    suppression_type = source.schema["suppressions"].dataType
    if isinstance(suppression_type, T.ArrayType) and isinstance(suppression_type.elementType, T.StructType):
        suppression_fields = suppression_type.elementType.fieldNames()

caveat_fields = []
if "caveats" in source.columns:
    caveat_type = source.schema["caveats"].dataType
    if isinstance(caveat_type, T.ArrayType) and isinstance(caveat_type.elementType, T.StructType):
        caveat_fields = caveat_type.elementType.fieldNames()

suppression_codes = F.transform(F.col("s.suppressions"), lambda x: x["suppression_code"]) if "suppression_code" in suppression_fields else F.lit(None).cast("array<string>")
suppression_names = F.transform(F.col("s.suppressions"), lambda x: x["suppression_name"]) if "suppression_name" in suppression_fields else F.lit(None).cast("array<string>")
suppression_display_values = F.transform(F.col("s.suppressions"), lambda x: x["suppression_display_value"]) if "suppression_display_value" in suppression_fields else F.lit(None).cast("array<string>")
suppression_footnotes = F.transform(F.col("s.suppressions"), lambda x: x["suppression_footnote"]) if "suppression_footnote" in suppression_fields else F.lit(None).cast("array<string>")

caveat_codes = F.transform(F.col("s.caveats"), lambda x: x["caveat_code"]) if "caveat_code" in caveat_fields else F.lit(None).cast("array<string>")
caveat_names = F.transform(F.col("s.caveats"), lambda x: x["caveat_name"]) if "caveat_name" in caveat_fields else F.lit(None).cast("array<string>")
caveat_display_values = F.transform(F.col("s.caveats"), lambda x: x["caveat_display_value"]) if "caveat_display_value" in caveat_fields else F.lit(None).cast("array<string>")
caveat_footnotes = F.transform(F.col("s.caveats"), lambda x: x["caveat_footnote"]) if "caveat_footnote" in caveat_fields else F.lit(None).cast("array<string>")

# Delimited text columns are easier to inspect in SQL and reporting tools than arrays.
suppression_reason_text = (
    F.concat_ws(" | ", F.transform(F.col("s.suppressions"), lambda x: F.concat_ws(
        " — ", x["suppression_code"], x["suppression_name"],
        x["suppression_display_value"], x["suppression_footnote"]
    ))) if {"suppression_code", "suppression_name", "suppression_display_value", "suppression_footnote"}.issubset(set(suppression_fields))
    else F.lit(None).cast("string")
)
caveat_reason_text = (
    F.concat_ws(" | ", F.transform(F.col("s.caveats"), lambda x: F.concat_ws(
        " — ", x["caveat_code"], x["caveat_name"],
        x["caveat_display_value"], x["caveat_footnote"]
    ))) if {"caveat_code", "caveat_name", "caveat_display_value", "caveat_footnote"}.issubset(set(caveat_fields))
    else F.lit(None).cast("string")
)

fact_measure_value = j.select(
    F.coalesce(F.col("u.reporting_unit_key"), F.lit(-1)).cast("long").alias("reporting_unit_key"),
    F.coalesce(F.col("m.measure_key"), F.lit(-1)).cast("long").alias("measure_key"),
    F.coalesce(F.col("p.period_key"), F.lit(-1)).cast("long").alias("period_key"),
    F.coalesce(F.col("r.reported_measure_key"), F.lit(-1)).cast("long").alias("reported_measure_key"),
    F.col("s.data_set_id").cast("long").alias("data_set_id"),
    F.col("s.source_measure_code").cast("string").alias("source_measure_code") if "source_measure_code" in source.columns else F.col("s.measure_code").cast("string").alias("source_measure_code"),
    F.col("s.measure_code").cast("string").alias("measure_code"),
    F.col("s.reported_measure_code").cast("string").alias("reported_measure_code"),
    F.col("s.reporting_unit_code").cast("string").alias("reporting_unit_code"),
    F.col("s.reporting_unit_type_code").cast("string").alias("reporting_unit_type_code") if "reporting_unit_type_code" in source.columns else F.lit(None).cast("string").alias("reporting_unit_type_code"),
    F.col("s.group_number").cast("integer").alias("group_number") if "group_number" in source.columns else F.lit(None).cast("integer").alias("group_number"),
    F.col("s.value").cast("double").alias("value"),
    F.col("s.lower_value").cast("double").alias("lower_value") if "lower_value" in source.columns else F.lit(None).cast("double").alias("lower_value"),
    F.col("s.upper_value").cast("double").alias("upper_value") if "upper_value" in source.columns else F.lit(None).cast("double").alias("upper_value"),
    suppression_count.alias("suppression_count"), caveat_count.alias("caveat_count"),
    is_suppressed.alias("is_suppressed"), is_missing.alias("is_value_missing"),
    suppression_details.alias("suppression_details"),
    suppression_codes.alias("suppression_codes"),
    suppression_names.alias("suppression_names"),
    suppression_display_values.alias("suppression_display_values"),
    suppression_footnotes.alias("suppression_footnotes"),
    caveat_details.alias("caveat_details"),
    caveat_codes.alias("caveat_codes"),
    caveat_names.alias("caveat_names"),
    caveat_display_values.alias("caveat_display_values"),
    caveat_footnotes.alias("caveat_footnotes"),
    suppression_reason_text.alias("suppression_reason_text"),
    caveat_reason_text.alias("caveat_reason_text"),
    F.col("s._period_start_date").alias("reporting_start_date"), F.col("s._period_end_date").alias("reporting_end_date"),
    F.current_timestamp().alias("gold_processed_at_utc")
)
print(f"Prepared Gold fact rows: {fact_measure_value.count():,}")
display(fact_measure_value.limit(10))


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 9, Finished, Available, Finished, False)

Silver fact-source rows: 1,709,311
Rows after dimension lookup joins: 1,709,311
Non-null business keys without dimension matches: {'reporting_units': 0, 'measures': 0, 'reported_measures': 0, 'periods': 0}
Prepared Gold fact rows: 1,709,311


SynapseWidget(Synapse.DataFrame, da024636-ac76-4ab6-9fb2-d23ea00d3d79)

In [8]:
# Check dimension keys, fact row counts, foreign keys and source metadata.

for df, key, label in [
    (dim_reporting_unit, "reporting_unit_key", "Reporting-unit dimension"),
    (dim_measure, "measure_key", "Measure dimension"),
    (dim_period, "period_key", "Period dimension"),
    (dim_reported_measure, "reported_measure_key", "Reported-measure dimension")
]:
    assert_unique(df, [key], label)

if fact_measure_value.count() != source_count:
    raise ValueError("Gold fact row count does not reconcile to Silver.")

for dim_name, key_col, dim_df in [
    (GOLD_REPORTING_UNITS, "reporting_unit_key", dim_reporting_unit),
    (GOLD_MEASURES, "measure_key", dim_measure),
    (GOLD_PERIODS, "period_key", dim_period),
    (GOLD_REPORTED_MEASURES, "reported_measure_key", dim_reported_measure)
]:
    orphan_count = fact_measure_value.select(key_col).distinct().join(dim_df.select(key_col).distinct(), key_col, "left_anti").count()
    print(f"{dim_name} orphan key count: {orphan_count}")
    if orphan_count:
        raise ValueError(f"Fact contains orphan {key_col} values.")

print("=== Fact value and suppression profile ===")
display(fact_measure_value.agg(
    F.count("*").alias("fact_rows"),
    F.sum(F.col("value").isNull().cast("long")).alias("null_value_rows"),
    F.sum(F.col("is_suppressed").cast("long")).alias("suppressed_rows"),
    F.sum(F.col("is_value_missing").cast("long")).alias("missing_value_rows"),
    F.sum((F.col("reporting_unit_key") == -1).cast("long")).alias("unknown_reporting_unit_rows"),
    F.sum((F.col("measure_key") == -1).cast("long")).alias("unknown_measure_rows"),
    F.sum((F.col("period_key") == -1).cast("long")).alias("unknown_period_rows"),
    F.sum((F.col("reported_measure_key") == -1).cast("long")).alias("unknown_reported_measure_rows")
))

print("=== Reported-measure variants ===")
display(dim_reported_measure.groupBy("is_total").count())
variant_diagnostic = fact_measure_value.filter(F.col("value").isNotNull()).groupBy(
    "measure_code", "reporting_unit_code", "period_key"
).agg(F.countDistinct("reported_measure_code").alias("reported_variant_count")).filter(F.col("reported_variant_count") > 1)
print(f"Measure/unit/period combinations with multiple reported variants: {variant_diagnostic.count():,}")
display(variant_diagnostic.orderBy(F.col("reported_variant_count").desc()).limit(20))


# Inspect real reason codes and labels before building report filters.
if "suppression_codes" in fact_measure_value.columns:
    print("=== Suppression code and label examples ===")
    display(
        fact_measure_value
        .filter(F.size(F.coalesce(F.col("suppression_codes"), F.array().cast("array<string>"))) > 0)
        .select("measure_code", "reported_measure_code", "value", "suppression_codes",
                "suppression_names", "suppression_display_values", "suppression_footnotes")
        .limit(20)
    )

if "caveat_codes" in fact_measure_value.columns:
    print("=== Caveat code and label examples ===")
    display(
        fact_measure_value
        .filter(F.size(F.coalesce(F.col("caveat_codes"), F.array().cast("array<string>"))) > 0)
        .select("measure_code", "reported_measure_code", "value", "caveat_codes",
                "caveat_names", "caveat_display_values", "caveat_footnotes")
        .limit(20)
    )


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 10, Finished, Available, Finished, False)

gold_dim_reporting_unit orphan key count: 0
gold_dim_measure orphan key count: 0
gold_dim_period orphan key count: 0
gold_dim_reported_measure orphan key count: 0
=== Fact value and suppression profile ===


SynapseWidget(Synapse.DataFrame, 910f7410-8e98-45f5-ac7b-c0606965385b)

=== Reported-measure variants ===


SynapseWidget(Synapse.DataFrame, 4b88412b-39a3-439e-bb52-a2114b8c1b8d)

Measure/unit/period combinations with multiple reported variants: 108,436


SynapseWidget(Synapse.DataFrame, 93e4b56f-52cc-46ec-b83b-b4da1f6ad30a)

=== Suppression code and label examples ===


SynapseWidget(Synapse.DataFrame, e745eefa-cd82-4389-8b8b-5e126ca3d250)

=== Caveat code and label examples ===


SynapseWidget(Synapse.DataFrame, ca16c212-4678-4166-ad46-b573d05cdb79)

In [9]:
# Write the four dimensions and fact table to the Lakehouse, then check row counts.

gold_dimensions = [
    (GOLD_REPORTING_UNITS, dim_reporting_unit),
    (GOLD_MEASURES, dim_measure),
    (GOLD_PERIODS, dim_period),
    (GOLD_REPORTED_MEASURES, dim_reported_measure)
]
expected_counts = {}
for table_name, dataframe in gold_dimensions:
    expected_counts[table_name] = dataframe.count()
    dataframe.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"Wrote {table_name}: {expected_counts[table_name]:,} rows")

expected_counts[GOLD_FACT] = fact_measure_value.count()
fact_measure_value.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD_FACT)
print(f"Wrote {GOLD_FACT}: {expected_counts[GOLD_FACT]:,} rows")

print("=== Persisted Gold row-count reconciliation ===")
for table_name, expected in expected_counts.items():
    actual = spark.table(table_name).count()
    print(f"{table_name}: expected {expected:,} | persisted {actual:,}")
    if actual != expected:
        raise ValueError(f"Persisted row count mismatch for {table_name}.")
print("Gold modeling completed successfully.")


StatementMeta(, d0991e86-5e7e-44e7-a180-6fdae899548b, 11, Finished, Available, Finished, False)

Wrote gold_dim_reporting_unit: 1,429 rows
Wrote gold_dim_measure: 34 rows
Wrote gold_dim_period: 61 rows
Wrote gold_dim_reported_measure: 691 rows
Wrote gold_fact_measure_value: 1,709,311 rows
=== Persisted Gold row-count reconciliation ===
gold_dim_reporting_unit: expected 1,429 | persisted 1,429
gold_dim_measure: expected 34 | persisted 34
gold_dim_period: expected 61 | persisted 61
gold_dim_reported_measure: expected 691 | persisted 691
gold_fact_measure_value: expected 1,709,311 | persisted 1,709,311
Gold modeling completed successfully.
